# Ollama Demo

### Importing Libraries

This notebook demonstrates how we will use Ollama to answer questions with context from Qdrant. We begin by importing the neccesary libraires

In [46]:
from qdrant_client import QdrantClient
from qdrant_client.http.models import PointStruct, VectorParams, Distance, QueryResponse
from ollama import Client as OllamaClient, ChatResponse
from typing import Mapping, Any, Optional, Sequence
import os
import hashlib

### Client Setup

Next we define functions to get Ollama and Qdrant clients

In [47]:
_ollama_client: Optional["OllamaClient"] = None

def _get_ollama_client() -> OllamaClient:
    global _ollama_client
    if _ollama_client is None:
        host = os.getenv("OLLAMA_HOST", "http://localhost:11434")
        _ollama_client = OllamaClient(host=host)
    return _ollama_client

In [48]:
_qdrant_client: Optional["QdrantClient"] = None

def _get_qdrant_client() -> QdrantClient:
    global _qdrant_client
    if _qdrant_client is None:
        url = os.getenv("QDRANT_HOST", "http://localhost:6333")
        _qdrant_client = QdrantClient(url=url)
    return _qdrant_client

### RAG Helper Functions

Next, we define a family of function to run a "RAG" query against a pre-existing Qdrant collection. We need [1] a function to embded a query, [2]a function to search for similar strings in qdrant, [3] a function to build our prompt with instructions and context, and [4] a function to send our prompt to Ollama

In [49]:
def embed(text:str) -> list[float]:
    _ollama_client = _get_ollama_client()
    response: ChatResponse = _ollama_client.embed(model="nomic-embed-text", input=text) # type: ignore
    return response.embeddings[0] # type: ignore

In [50]:
def search(vector: Sequence[float], collection_name: str, limit: int) -> Sequence[str]:
    qdrant_client = _get_qdrant_client()
    results: QueryResponse = qdrant_client.query_points(
        collection_name=collection_name,
        query=vector,
        limit=limit,
    )

    return [results.points[i].payload["text"] for i in range(len(results.points))]

In [51]:
system_prompt = """You are an AI assistant that helps people find information.
Use the provided context to answer the question."""

system_prompt

'You are an AI assistant that helps people find information.\nUse the provided context to answer the question.'

In [52]:
def build_prompt(user_input: str, context: Sequence[str], system_prompt: str) -> Sequence[Mapping[str, Any]]:
    context_block = "\n".join(
        f"[{idx+1}] {ctx}"
        for idx, ctx in enumerate(context)
    )

    prompt = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"CONTEXT:\n{context_block}"},
        {"role": "user", "content": f"QUESTION:\n{user_input}"},
    ]
    return prompt


prompt = build_prompt("Where is France and what is its capital?", ["France is a country in Europe", "Its capital is Paris."], system_prompt)
print(prompt, "\n")
for i in prompt:
    print(f"role = {i['role']}:")
    print(i["content"])
    print("\n")

[{'role': 'system', 'content': 'You are an AI assistant that helps people find information.\nUse the provided context to answer the question.'}, {'role': 'user', 'content': 'CONTEXT:\n[1] France is a country in Europe\n[2] Its capital is Paris.'}, {'role': 'user', 'content': 'QUESTION:\nWhere is France and what is its capital?'}] 

role = system:
You are an AI assistant that helps people find information.
Use the provided context to answer the question.


role = user:
CONTEXT:
[1] France is a country in Europe
[2] Its capital is Paris.


role = user:
QUESTION:
Where is France and what is its capital?




In [53]:
def query(messages: Sequence[Mapping[str, Any]], model: str = "llama3.2") -> ChatResponse:
    """
    messages: sequence of dicts or Message objects, e.g. {"role": "user", "content": "Hello"}
    Returns a ChatResponse (stream=False).
    """
    # Basic structural validation
    for m in messages:
        if isinstance(m, dict):
            if "role" not in m or "content" not in m:
                raise ValueError("message dict must contain 'role' and 'content' keys")

    ollama_client = _get_ollama_client()
    response: ChatResponse = ollama_client.chat(model="llama3.2", messages=messages, stream=False) # type: ignore

    return response

### Embedding Context with Qdrant

Now we embed context into a Qdrant collection:

First we define a function to hash each document in order to create a unique ID

In [54]:
def document_id(document: str, truncate: int = 32) -> str:
    normalized = " ".join(document.split()).strip().lower()
    encoded = normalized.encode('utf-8')
    hash = hashlib.sha256()
    hash.update(encoded)
    return hash.hexdigest()[:truncate]

This function writes each document to Qdrant:

In [55]:
def qdrant_write(document: str, collection: str) -> None:
    vector: list[float] = embed(document)
    point = PointStruct(
        id=document_id(document), #automatically translated to UUID
        vector=vector,
        payload={"text": document}
        )
    
    qdrant_client = _get_qdrant_client()

    if qdrant_client.collection_exists(collection_name=collection):
        pass
    else:
        qdrant_client.create_collection(
            collection_name=collection,
            vectors_config=VectorParams(
                size=len(vector),
                distance=Distance.COSINE
            )
        )

    qdrant_client.upsert(
        collection_name=collection,
        points=[point],
    )

Now we test our embedding process:

In [56]:
qdrant_write("France is a country in Europe. Its capital is Paris.", "test_collection")

qdrant_client = _get_qdrant_client()
qdrant_client.scroll(collection_name="test_collection")

([Record(id='a284ca9d-2b50-ce30-098f-8c5d568a55cd', payload={'text': 'France is a country in Europe. Its capital is Paris.'}, vector=None, shard_key=None, order_value=None)],
 None)

We can also clear a collection if necessary

In [57]:
#qdrant_client = _get_qdrant_client()
#qdrant_client.delete_collection(collection_name="test_collection")

### RAG Query

Finally, we combine our RAG helper functions to perform a RAG query

In [58]:
user_query = "Where is France and what is its capital?"

In [59]:
vector = embed(user_query)

In [62]:
context = search(vector, collection_name="test_collection", limit=1)
context

['France is a country in Europe. Its capital is Paris.']

In [63]:
prompt = build_prompt(user_query, context, system_prompt)
prompt

[{'role': 'system',
  'content': 'You are an AI assistant that helps people find information.\nUse the provided context to answer the question.'},
 {'role': 'user',
  'content': 'CONTEXT:\n[1] France is a country in Europe. Its capital is Paris.'},
 {'role': 'user',
  'content': 'QUESTION:\nWhere is France and what is its capital?'}]

In [64]:
query_response = query(prompt, model="llama3.2")
query_response

ChatResponse(model='llama3.2', created_at='2026-03-03T19:42:32.191819125Z', done=True, done_reason='stop', total_duration=4821817543, load_duration=2212208251, prompt_eval_count=73, prompt_eval_duration=1359233624, eval_count=21, eval_duration=1239455751, message=Message(role='assistant', content='According to the provided context, France:\n\n* Is located in Europe\n* Has its capital in Paris', thinking=None, images=None, tool_name=None, tool_calls=None), logprobs=None)

In [70]:
print(query_response.message.content)

According to the provided context, France:

* Is located in Europe
* Has its capital in Paris
